# Phase 6 — Abstractive Summarization · Kaggle 2× T4

**Inputs:** `prefixlm-run1` (tokenizer) + `phase1-models`. Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN`.

**Yours (rule 17.5):** `sum_model.py` (Phase-1 backbone WITH its LM head, adaptation methods, `logits()` and optionally
`prefill()`/`decode_step()`) and every `TODO` (source/summary budgets, separator, decoding, training settings).
**Harness:** held-out validation split, token cache, source selection, summary-only loss, decoding loop, ROUGE (+ optional
BERTScore), teacher-summary KD, `[{"id","summary"}]` + zip.

Phase-6 technique list (doc 12.2.1): Houlsby adapters · Pfeiffer adapters · Prefix tuning · Prompt tuning · LoRA · QLoRA · AdaLoRA · **LoRMA** · **KD**.
(Full and partial fine-tuning are NOT on this list.)

Order: cells 1-6 (tests pass, data check, SMOKE TEST PASSED) → paste `sum_model.py` → plan → model check → run.

In [ ]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
NB_START = time.time()
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

In [ ]:
# 2 — Phase-6 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P6 = {}
for f in ["Train/train.json", "Test/test.json"]:
    P6[f.split("/")[-1].replace(".json", "")] = hf_hub_download(REPO, f"Phase_6_Summ/{f}", repo_type="dataset")
print(json.dumps(P6, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")

In [ ]:
%%writefile sum_data.py
"""sum_data.py — Phase-6 abstractive summarization data handling."""
import hashlib, json, os, pickle, random, re
from collections import Counter

import numpy as np

ID_KEYS = ("id", "doc_id", "case_id", "uid", "name")
TEXT_KEYS = ("document", "text", "judgment", "judgement", "source", "doc")
SUM_KEYS = ("summary", "abstract", "headnote", "target", "reference")


def _get(rec, keys):
    for k in keys:
        if k in rec:
            return rec[k]
    return None


def _join(v):
    return "\n".join(str(x) for x in v) if isinstance(v, (list, tuple)) else (None if v is None else str(v))


def load_cases(path):
    obj = json.load(open(path, encoding="utf-8"))
    if isinstance(obj, dict) and not any(k in obj for k in ID_KEYS):
        for k in ("data", "cases", "examples", "instances"):
            if isinstance(obj.get(k), list):
                obj = obj[k]
                break
    recs = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            recs.append(dict(id=k, document=v) if isinstance(v, str) else dict(v, id=v.get("id", k)))
    else:
        recs = list(obj)
    out, dup = {}, 0
    for i, r in enumerate(recs):
        cid = _get(r, ID_KEYS)
        cid = i if cid is None else cid
        text = _join(_get(r, TEXT_KEYS))
        assert text is not None, f"record {cid}: no document field among {TEXT_KEYS}; keys {list(r)[:8]}"
        key = str(cid)
        if key in out:
            dup += 1
            key = f"{key}#{dup}"
        out[key] = dict(id=cid, text=text, summary=_join(_get(r, SUM_KEYS)))
    return out


def split_train_val(keys, val_cases, seed=0):
    keys = sorted(keys)
    rng = random.Random(seed)
    val = set(rng.sample(keys, min(len(keys) // 2, int(val_cases)))) if val_cases else set()
    return [k for k in keys if k not in val], [k for k in keys if k in val]


def tokenize_texts(texts, encode_batch, cache_dir=None, tag=""):
    h = hashlib.md5((tag + "|".join(f"{k}:{len(v)}" for k, v in texts.items())).encode()).hexdigest()[:16]
    path = os.path.join(cache_dir, f"sum_tok_{h}.pkl") if cache_dir else None
    if path and os.path.exists(path):
        return pickle.load(open(path, "rb"))
    keys = list(texts)
    out = {}
    for i in range(0, len(keys), 256):
        ks = keys[i:i + 256]
        for k, ids in zip(ks, encode_batch([texts[k] for k in ks])):
            out[k] = np.asarray(ids, dtype=np.uint16)
    if path:
        os.makedirs(cache_dir, exist_ok=True)
        tmp = path + f".tmp{os.getpid()}"
        pickle.dump(out, open(tmp, "wb"), protocol=4)
        os.replace(tmp, path)
    return out


def select_src(ids, max_src, mode="first", head_frac=0.5):
    n = len(ids)
    if n <= max_src:
        return ids
    if mode == "first":
        return ids[:max_src]
    if mode == "last":
        return ids[n - max_src:]
    if mode == "head_tail":
        h = int(round(max_src * float(head_frac)))
        return np.concatenate([ids[:h], ids[n - (max_src - h):]]) if max_src - h else ids[:h]
    raise ValueError(f"src_select must be first | last | head_tail, got {mode!r}")


_W = re.compile(r"[a-z0-9]+")


def _words(t):
    return _W.findall(t.lower())


def _ngrams(w, n):
    return Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))


def _f(overlap, p_tot, r_tot):
    if not overlap or not p_tot or not r_tot:
        return 0.0
    p, r = overlap / p_tot, overlap / r_tot
    return 2 * p * r / (p + r)


def _lcs(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_pair(pred, ref):
    p, r = _words(pred), _words(ref)
    out = {}
    for n in (1, 2):
        pn, rn = _ngrams(p, n), _ngrams(r, n)
        out[f"rouge{n}"] = _f(sum((pn & rn).values()), sum(pn.values()), sum(rn.values()))
    out["rougeL"] = _f(_lcs(p, r), len(p), len(r))
    return out


def rouge(preds, refs):
    rows = [rouge_pair(p, r) for p, r in zip(preds, refs)]
    return {k: float(np.mean([x[k] for x in rows])) if rows else 0.0 for k in ("rouge1", "rouge2", "rougeL")}


def validate_prediction(pred_list, test_cases):
    assert isinstance(pred_list, list), "prediction.json must be a JSON list"
    ids = [str(r["id"]) for r in pred_list]
    assert len(ids) == len(set(ids)), "duplicate ids in prediction"
    want = {str(v["id"]) for v in test_cases.values()}
    assert set(ids) == want, f"ids differ from test.json: missing {len(want - set(ids))}, extra {len(set(ids) - want)}"
    for r in pred_list:
        assert set(r) == {"id", "summary"}, f"each entry must be exactly {{id, summary}}, got {sorted(r)}"
        assert isinstance(r["summary"], str), f"summary must be a string for {r['id']}"
    empty = sum(1 for r in pred_list if not r["summary"].strip())
    return len(pred_list), empty


def length_stats(toks):
    n = sorted(len(v) for v in toks.values())
    q = lambda f: int(n[min(len(n) - 1, int(len(n) * f))]) if n else 0
    return dict(n=len(n), p10=q(.1), p50=q(.5), p90=q(.9), p99=q(.99), max=int(n[-1]) if n else 0)

In [ ]:
%%writefile test_sum_data.py
"""Synthetic tests for sum_data.py (python test_sum_data.py)."""
import json, tempfile

import numpy as np

import sum_data as sd

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


tmp = tempfile.mkdtemp()
rows = [dict(id=f"d{i}", document=f"doc text {i}", summary=f"sum {i}") for i in range(10)]
json.dump(rows, open(f"{tmp}/a.json", "w"))
json.dump({r["id"]: dict(text=r["document"], summary=r["summary"]) for r in rows}, open(f"{tmp}/b.json", "w"))
json.dump([dict(id=r["id"], document=r["document"]) for r in rows], open(f"{tmp}/t.json", "w"))
a, b, t = (sd.load_cases(f"{tmp}/{x}.json") for x in "abt")
check("1 loaders agree", all(a[k]["text"] == b[k]["text"] and a[k]["summary"] == b[k]["summary"] for k in a))
check("1 test has no summary", all(v["summary"] is None for v in t.values()))


tr, va = sd.split_train_val(list(a), 3, seed=1)
tr2, va2 = sd.split_train_val(list(a), 3, seed=1)
check("2 split disjoint + deterministic", len(va) == 3 and not set(tr) & set(va) and va == va2 and len(tr) == 7)


ids = np.arange(100)
check("3 first/last", list(sd.select_src(ids, 10, "first")) == list(range(10)) and list(sd.select_src(ids, 10, "last")) == list(range(90, 100)))
check("3 head_tail", list(sd.select_src(ids, 10, "head_tail", 0.3)) == [0, 1, 2] + list(range(93, 100)))
check("3 short doc unchanged", len(sd.select_src(np.arange(5), 10, "last")) == 5)


r = sd.rouge_pair("the cat sat on the mat", "the cat sat on a mat")

check("4 rouge1", abs(r["rouge1"] - 5 / 6) < 1e-9)
check("4 rougeL", abs(r["rougeL"] - 5 / 6) < 1e-9)
check("4 rouge2", abs(r["rouge2"] - 3 / 5) < 1e-9)
check("4 identical = 1, disjoint = 0", sd.rouge_pair("a b c", "a b c")["rougeL"] == 1.0 and sd.rouge_pair("a b", "c d")["rouge1"] == 0.0)
check("4 case/punctuation ignored", sd.rouge_pair("The Court, held.", "the court held")["rouge1"] == 1.0)


good = [dict(id=v["id"], summary="x") for v in t.values()]
check("5 accepts", sd.validate_prediction(good, t) == (10, 0))
for bad, why in [(good[:-1], "missing id"), (good + [good[0]], "duplicate"), ([dict(r, extra=1) for r in good], "extra key"),
                 ([dict(id=r["id"], summary=None) for r in good], "non-string")]:
    try:
        sd.validate_prediction(bad, t); check(f"5 rejects {why}", False)
    except AssertionError:
        check(f"5 rejects {why}", True)

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

In [ ]:
%%writefile sum_train.py
"""sum_train.py — Phase-6 abstractive summarization harness."""
import importlib, json, math, os, random, re, sys, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F

import sum_data as sd

CFG = json.load(open(sys.argv[1]))
D, TR, G, MODEL_CFG = CFG["data"], CFG["train"], CFG.get("gen", {}), CFG["model"]
KD = CFG.get("kd") or {}
OUT, NAME = CFG["out_dir"], CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ tokenizer + data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))
V = TOK.get_vocab_size()
PAD = TOK.token_to_id("<pad>") or 0
EOS = TOK.token_to_id(D.get("end_token", "</s>"))
START = TOK.token_to_id(D.get("start_token", "</s>"))
assert EOS is not None and START is not None, "start/end token not in the vocabulary"
SEP_IDS = np.asarray(TOK.encode(D["sep_text"]).ids, np.uint16) if D.get("sep_text") else np.zeros(0, np.uint16)
MAXS, MAXT = int(D["max_src_tokens"]), int(D["max_tgt_tokens"])
SEL, HFRAC = D.get("src_select", "first"), float(D.get("head_frac", 0.5))
BAN = [i for i in (TOK.token_to_id(t) for t in ("<s>", "<unk>", "<pad>", "<mask>")) if i is not None and i != EOS]


def enc_batch(texts):
    return [e.ids for e in TOK.encode_batch(texts)]


ALL = sd.load_cases(D["train"])
TEST = sd.load_cases(D["test"]) if D.get("test") else {}
_nosum = [k for k, v in ALL.items() if not (v["summary"] or "").strip()]
for _k in _nosum:
    del ALL[_k]
assert ALL, "no training documents with a summary"
TR_KEYS, VA_KEYS = sd.split_train_val(list(ALL), int(D.get("val_cases", 0)), int(D.get("val_seed", 0)))
cache = D.get("cache_dir", "sum_cache")
DOC = sd.tokenize_texts({k: v["text"] for k, v in ALL.items()}, enc_batch, cache, "train-doc")
SUMT = sd.tokenize_texts({k: v["summary"] for k, v in ALL.items()}, enc_batch, cache, "train-sum")
if TEST:
    DOC.update({f"test::{k}": v for k, v in sd.tokenize_texts({k: v["text"] for k, v in TEST.items()}, enc_batch, cache, "test-doc").items()})

TARGET = {k: SUMT[k] for k in TR_KEYS}
if KD.get("teacher_summaries"):
    tsum = json.load(open(KD["teacher_summaries"], encoding="utf-8"))["summaries"]
    tsum = {k: v for k, v in tsum.items() if k in ALL and k not in VA_KEYS}
    assert tsum, "teacher summaries cover none of the training documents"
    tt = {k: np.asarray(ids, np.uint16) for k, ids in zip(tsum, enc_batch(list(tsum.values())))}
    if KD.get("mode", "replace") == "replace":
        TR_KEYS = [k for k in TR_KEYS if k in tt]
        TARGET = {k: tt[k] for k in TR_KEYS}
    else:
        for k, v in tt.items():
            TARGET[f"kd::{k}"] = v
        TR_KEYS = TR_KEYS + [f"kd::{k}" for k in tt]
    log(f"KD: {len(tt)} teacher summaries ({KD.get('mode', 'replace')}) -> {len(TR_KEYS)} training pairs")
if TR.get("train_subset"):
    TR_KEYS = random.Random(SEED).sample(TR_KEYS, min(len(TR_KEYS), int(TR["train_subset"])))


def doc_key(k):
    return k[4:] if k.startswith("kd::") else k


def src_of(k):
    body = sd.select_src(DOC[doc_key(k)], MAXS, SEL, HFRAC)
    if not len(body):
        body = np.asarray([START], np.uint16)
    return np.concatenate([body, SEP_IDS])


def tgt_of(ids):
    return np.concatenate([ids[:MAXT], np.asarray([EOS], np.uint16)])


ls = lambda d: sd.length_stats(d)
log(f"dropped {len(_nosum)} training documents with an empty summary: {_nosum[:5]}")
log(f"train pairs {len(TR_KEYS)} | val {len(VA_KEYS)} | test {len(TEST)} | doc tokens {ls({k: DOC[k] for k in ALL})} | "
    f"summary tokens {ls(SUMT)} | max_src {MAXS} ({SEL}) + sep {len(SEP_IDS)} | max_tgt {MAXT} | "
    f"summaries longer than max_tgt: {sum(1 for v in SUMT.values() if len(v) > MAXT) / len(SUMT):.1%}")


def batch_src(keys):
    srcs = [src_of(k) for k in keys]
    m = min(len(s) for s in srcs)
    rows = []
    for s in srcs:
        body = s[:len(s) - len(SEP_IDS)]
        keep = m - len(SEP_IDS)
        body = body[len(body) - keep:] if SEL == "last" else body[:keep]
        rows.append(np.concatenate([body, SEP_IDS]))
    return torch.tensor(np.stack(rows).astype(np.int64))


def batch_tgt(tgts):
    T = max(len(t) for t in tgts)
    tin = torch.full((len(tgts), T), PAD, dtype=torch.long)
    lab = torch.full((len(tgts), T), -100, dtype=torch.long)
    for i, t in enumerate(tgts):
        t = torch.tensor(t.astype(np.int64))
        tin[i, 0] = START
        tin[i, 1:len(t)] = t[:-1]
        lab[i, :len(t)] = t
    return tin, lab


def by_src_len(keys):
    return sorted(keys, key=lambda k: len(src_of(k)))


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "sum_model"))
model = model_lib.build_sum_model(dict(MODEL_CFG)).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
assert n_train > 0, "no trainable parameters"
HAS_CACHE = hasattr(model, "prefill") and hasattr(model, "decode_step")
log("generation: " + ("cached (prefill/decode_step)" if HAS_CACHE else "no cache: logits() recomputed each step (slower)"))
if hasattr(model, "max_total_len"):
    assert model.max_total_len(MAXS + len(SEP_IDS), MAXT + 1) is not False, "max_src_tokens + max_tgt_tokens do not fit the model"

if CFG.get("load_trained"):
    sd_ = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]
    cur = dict(model.named_parameters())
    miss = [k for k in sd_ if k not in cur]
    assert not miss, f"load_trained: names not in this model, e.g. {miss[:3]}"
    with torch.no_grad():
        for k, v in sd_.items():
            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))
    log(f"loaded {len(sd_)} trained tensors from {CFG['load_trained']}")


def step_loss(keys, tgt_map):
    src = batch_src(keys).to(DEV)
    tin, lab = batch_tgt([tgt_of(tgt_map[k]) for k in keys])
    tin, lab = tin.to(DEV), lab.to(DEV)
    with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
        lg = model.logits(src, tin)
    assert tuple(lg.shape) == (len(keys), tin.shape[1], V), f"logits {tuple(lg.shape)}, expected {(len(keys), tin.shape[1], V)}"
    loss = F.cross_entropy(lg.float().reshape(-1, V), lab.reshape(-1), ignore_index=-100,
                           label_smoothing=float(TR.get("label_smoothing", 0.0)))
    return loss, int((lab != -100).sum())


@torch.no_grad()
def val_loss(keys):
    model.eval()
    tot, n = 0.0, 0
    ks = by_src_len(keys)
    bs = int(TR.get("eval_batch_size", TR["batch_size"]))
    for i in range(0, len(ks), bs):
        l, c = step_loss(ks[i:i + bs], SUMT)
        tot += l.item() * c; n += c
    model.train()
    return tot / max(1, n)


# ------------------------------------------------------------------ generation
class NgramBlocker:
    def __init__(self, n):
        self.n, self.seen, self.hist = n, {}, []

    def banned(self):
        if not self.n or len(self.hist) < self.n - 1:
            return ()
        return self.seen.get(tuple(self.hist[-(self.n - 1):]), ())

    def push(self, t):
        if self.n and len(self.hist) >= self.n - 1:
            self.seen.setdefault(tuple(self.hist[-(self.n - 1):]), set()).add(t)
        self.hist.append(t)


def pick_next(logits, gen, blockers, step):
    logits = logits.float()
    if BAN:
        logits[:, BAN] = -float("inf")
    if step < int(G.get("min_new_tokens", 0)):
        logits[:, EOS] = -float("inf")
    rp = float(G.get("rep_penalty", 1.0) or 1.0)
    if rp != 1.0 and gen.numel():
        sc = torch.gather(logits, 1, gen)
        logits.scatter_(1, gen, torch.where(sc < 0, sc * rp, sc / rp))
    for b, bl in enumerate(blockers):
        bad = bl.banned()
        if bad:
            logits[b, list(bad)] = -float("inf")
    T = float(G.get("temperature", 0.0) or 0.0)
    if not T:
        return logits.argmax(-1)
    logits = logits / T
    k = int(G.get("top_k", 0) or 0)
    if k:
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    p = float(G.get("top_p", 1.0) or 1.0)
    probs = F.softmax(logits, -1)
    if p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        sp[(sp.cumsum(-1) - sp) > p] = 0.0
        probs = torch.zeros_like(probs).scatter_(1, si, sp)
    return torch.multinomial(probs / probs.sum(-1, keepdim=True), 1).squeeze(1)


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


@torch.no_grad()
def generate(keys, src_fn):
    model.eval()
    out = {}
    ks = sorted(keys, key=lambda k: len(src_fn(k)))
    bs = int(G.get("batch_size", 8))
    max_new = int(G.get("max_new_tokens", MAXT))
    for i in range(0, len(ks), bs):
        b = ks[i:i + bs]
        srcs = [src_fn(k) for k in b]
        m = min(len(s) for s in srcs)
        rows = []
        for s in srcs:
            body, keep = s[:len(s) - len(SEP_IDS)], m - len(SEP_IDS)
            body = body[len(body) - keep:] if SEL == "last" else body[:keep]
            rows.append(np.concatenate([body, SEP_IDS]))
        src = torch.tensor(np.stack(rows).astype(np.int64), device=DEV)
        B = len(b)
        tin = torch.full((B, 1), START, dtype=torch.long, device=DEV)
        gen = torch.empty((B, 0), dtype=torch.long, device=DEV)
        done = torch.zeros(B, dtype=torch.bool, device=DEV)
        blockers = [NgramBlocker(int(G.get("no_repeat_ngram", 0) or 0)) for _ in range(B)]
        use_cache = HAS_CACHE
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            if use_cache:
                try:
                    nl, cachev = model.prefill(src, tin)
                except NotImplementedError:
                    use_cache = False
            if not use_cache:
                nl = model.logits(src, tin)[:, -1]
        for step in range(max_new):
            nxt = pick_next(nl, gen, blockers, step)
            nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
            gen = torch.cat([gen, nxt[:, None]], 1)
            for bb, t in enumerate(nxt.tolist()):
                blockers[bb].push(t)
            done |= nxt == EOS
            if bool(done.all()) or step == max_new - 1:
                break
            with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
                if use_cache:
                    nl, cachev = model.decode_step(nxt[:, None], cachev)
                else:
                    nl = model.logits(src, torch.cat([tin, gen], 1))[:, -1]
        for bb, k in enumerate(b):
            ids = gen[bb].tolist()
            ids = ids[:ids.index(EOS)] if EOS in ids else ids
            out[k] = clean(TOK.decode(ids))
    model.train()
    return out


def gen_scores(keys, tag):
    t1 = time.time()
    pred = generate(keys, src_of)
    refs = [ALL[k]["summary"] for k in keys]
    sc = sd.rouge([pred[k] for k in keys], refs)
    sc.update(words_pred=float(np.mean([len(pred[k].split()) for k in keys])) if keys else 0.0,
              words_ref=float(np.mean([len(r.split()) for r in refs])) if keys else 0.0,
              gen_seconds=time.time() - t1, n=len(keys))
    return sc, pred


# ------------------------------------------------------------------ training
bs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))
epochs = int(TR["epochs"])
per_epoch = min(len(TR_KEYS), int(TR.get("epoch_cases") or len(TR_KEYS)))
steps_per_epoch = math.ceil(per_epoch / (bs * acc))
total_steps = steps_per_epoch * epochs
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR["lr"])
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))
for g in opt.param_groups:
    g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
SELECT = TR.get("select_metric", "val_loss")
VGEN = random.Random(SEED + 3).sample(VA_KEYS, min(len(VA_KEYS), int(TR.get("val_gen_cases", 0) or 0)))
assert SELECT == "val_loss" or VGEN, "select_metric on ROUGE needs val_gen_cases > 0"
log(f"cases/epoch {per_epoch} | steps/epoch {steps_per_epoch} | epochs {epochs} | select by {SELECT} | "
    f"per-epoch generation on {len(VGEN)} val docs")


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    t = (step - warm) / max(1, total_steps - warm)
    sch = TR.get("schedule", "linear")
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


best, best_state, history = None, None, []
if epochs == 0:
    best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
order = []
model.train()
for ep in range(1, epochs + 1):
    if len(order) < per_epoch:
        rest = TR_KEYS[:]; random.shuffle(rest); order += rest
    ep_keys, order = order[:per_epoch], order[per_epoch:]

    ep_keys = by_src_len(ep_keys)
    batches = [ep_keys[i:i + bs] for i in range(0, len(ep_keys), bs)]
    random.shuffle(batches)
    run_loss, nb = 0.0, 0
    for b in batches:
        loss, _ = step_loss(b, TARGET)
        scaler.scale(loss / acc).backward()
        nb += 1; run_loss += loss.item()
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            break
    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)
    if VA_KEYS:
        rec["val_loss"] = val_loss(VA_KEYS)
    if VGEN:
        sc, _ = gen_scores(VGEN, "epoch")
        rec.update(val_rouge1=sc["rouge1"], val_rougeL=sc["rougeL"], val_words=sc["words_pred"])
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    score = rec.get(SELECT, ep if SELECT != "val_loss" else -ep)
    better = best is None or (score <= best if SELECT == "val_loss" else score >= best)
    if better:
        best = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        log(f"  best so far (epoch {ep})")
    if (time.time() - t0) / 60 > max_min:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0

cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG), os.path.join(OUT, f"{NAME}_trained_params.pt"))

result = dict(run=NAME, max_src_tokens=MAXS, src_select=SEL, max_tgt_tokens=MAXT, sep_text=D.get("sep_text"),
              params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb,
              gen=G, history=history, kd=KD or None)

# ------------------------------------------------------------------ final validation generation (+ optional BERTScore)
FV = random.Random(SEED + 5).sample(VA_KEYS, min(len(VA_KEYS), int(G.get("final_val_cases", 100) or 0)))
if FV:
    sc, vpred = gen_scores(FV, "final")
    if G.get("bertscore"):
        try:
            from bert_score import score as bscore
            P_, R_, F_ = bscore([vpred[k] for k in FV], [ALL[k]["summary"] for k in FV], lang="en",
                                model_type=G.get("bertscore_model"), batch_size=16, verbose=False)
            sc.update(bertscore_p=float(P_.mean()), bertscore_r=float(R_.mean()), bertscore_f1=float(F_.mean()))
        except Exception as e:
            log(f"BERTScore skipped: {type(e).__name__}: {e}")
    result["val"] = sc
    log("VAL generation: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in sc.items()))
    json.dump([dict(id=ALL[k]["id"], summary=vpred[k], reference=ALL[k]["summary"]) for k in FV],
              open(os.path.join(OUT, f"{NAME}_val_predictions.json"), "w", encoding="utf-8"), ensure_ascii=False)
    ex = FV[0]
    log(f"  example [{ALL[ex]['id']}]: {vpred[ex][:400]}")

# ------------------------------------------------------------------ teacher summaries for KD (training documents)
if CFG.get("dump_teacher_summaries"):
    pool = [k for k in sorted(ALL) if k not in set(VA_KEYS)]
    tk = random.Random(SEED + 11).sample(pool, min(len(pool), int(CFG["dump_teacher_summaries"])))
    t1 = time.time()
    tp = generate(tk, src_of)
    json.dump(dict(run=NAME, summaries=tp), open(os.path.join(OUT, f"{NAME}_teacher_summaries.json"), "w", encoding="utf-8"),
              ensure_ascii=False)
    log(f"teacher summaries written for {len(tp)} training documents ({(time.time() - t1) / 60:.1f} min)")

# ------------------------------------------------------------------ test
if TEST:
    t1 = time.time()
    tpred = generate([f"test::{k}" for k in TEST], src_of)
    pred = [dict(id=TEST[k]["id"], summary=tpred[f"test::{k}"]) for k in TEST]
    n, empty = sd.validate_prediction(pred, TEST)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    w = [len(r["summary"].split()) for r in pred]
    result.update(test_docs=n, test_empty=empty, test_words_mean=float(np.mean(w)), test_gen_seconds=time.time() - t1)
    log(f"test: {n} docs | words per summary mean {np.mean(w):.0f} (min {min(w)}, max {max(w)}) | empty {empty} | "
        f"generation {(time.time() - t1) / 60:.1f} min")
    log(f"wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1, default=str)
log(f"done: {NAME} | best {SELECT} {best if best is not None else float('nan'):.4f} | train {train_min:.1f} min | "
    f"peak {peak_gb:.1f} GB | trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

In [ ]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

In [ ]:
%%writefile sum_dummy.py
"""Smoke-test model only (NOT a submission model): bag-of-embeddings source + embedding of the previous token."""
import torch
import torch.nn as nn


class Dummy(nn.Module):
    def __init__(self, vocab_size, cached):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 32)
        self.out = nn.Linear(32, vocab_size)
        self.cached = cached

    def logits(self, src, tgt_in):
        ctx = self.emb(src).mean(1, keepdim=True)
        return self.out(ctx + self.emb(tgt_in))

    def _prefill(self, src, tgt_in):
        ctx = self.emb(src).mean(1)
        return self.out(ctx + self.emb(tgt_in[:, -1])), {"ctx": ctx}

    def _decode_step(self, next_ids, cache):
        return self.out(cache["ctx"] + self.emb(next_ids[:, -1])), cache

    def __getattr__(self, name):
        if name in ("prefill", "decode_step"):
            if self.__dict__.get("cached", False):
                return object.__getattribute__(self, "_" + name)
            raise AttributeError(name)
        return super().__getattr__(name)


def build_sum_model(model_cfg):
    return Dummy(int(model_cfg["vocab_size"]), bool(model_cfg.get("cached", False)))

In [ ]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

In [ ]:
# 3 — data check: layout, sizes, document + summary lengths (tokens and words), casing; builds the token caches
!python test_sum_data.py | tail -1
import sum_data as sd, importlib; importlib.reload(sd)
import numpy as np
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
V = _tok.get_vocab_size()
enc_batch = lambda texts: [e.ids for e in _tok.encode_batch(texts)]
for split in ("train", "test"):
    raw = json.load(open(P6[split]))
    ex = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{split}.json layout: {type(raw).__name__} | keys {list(ex)[:6] if isinstance(ex, dict) else '-'} | example: {str(ex)[:200]} ...")
TRAIN_ALL = sd.load_cases(P6["train"]); TEST_ALL = sd.load_cases(P6["test"])
t0 = time.time()
DOC_T = sd.tokenize_texts({k: v["text"] for k, v in TRAIN_ALL.items()}, enc_batch, "sum_cache", "train-doc")
SUM_T = sd.tokenize_texts({k: v["summary"] for k, v in TRAIN_ALL.items()}, enc_batch, "sum_cache", "train-sum")
TDOC_T = sd.tokenize_texts({k: v["text"] for k, v in TEST_ALL.items()}, enc_batch, "sum_cache", "test-doc")
print(f"tokenized in {time.time()-t0:.0f}s")
wq = lambda xs: dict(p10=int(np.percentile(xs, 10)), p50=int(np.percentile(xs, 50)), p90=int(np.percentile(xs, 90)), max=int(max(xs)))
print("train documents:", len(TRAIN_ALL), "| test documents:", len(TEST_ALL), "| missing summaries:", sum(1 for v in TRAIN_ALL.values() if not v["summary"]))
print("train doc tokens:", sd.length_stats(DOC_T), "| words:", wq([len(v["text"].split()) for v in TRAIN_ALL.values()]))
print("test  doc tokens:", sd.length_stats(TDOC_T), "| words:", wq([len(v["text"].split()) for v in TEST_ALL.values()]))
print("summary tokens:  ", sd.length_stats(SUM_T), "| words:", wq([len(v["summary"].split()) for v in TRAIN_ALL.values()]))
ratio = [len(SUM_T[k]) / max(1, len(DOC_T[k])) for k in TRAIN_ALL]
print("summary/document token ratio:", {f"p{q}": round(float(np.percentile(ratio, q)), 3) for q in (10, 50, 90)})
for S in (256, 512, 768):
    for Tt in (256, 512):
        print(f"  docs fully inside {S} source tokens: {np.mean([len(DOC_T[k]) <= S for k in TRAIN_ALL]):.0%} | "
              f"summaries inside {Tt} tokens: {np.mean([len(SUM_T[k]) <= Tt for k in TRAIN_ALL]):.0%}")
up = lambda xs: sum(sum(c.isupper() for c in t[:5000]) for t in xs) / max(1, sum(sum(c.isalpha() for c in t[:5000]) for t in xs))
print(f"uppercase share: train docs {up([v['text'] for v in TRAIN_ALL.values()]):.3f} | summaries {up([v['summary'] for v in TRAIN_ALL.values()]):.3f} | test docs {up([v['text'] for v in TEST_ALL.values()]):.3f}")
k0 = next(iter(TRAIN_ALL)); print("\nexample summary:", TRAIN_ALL[k0]["summary"][:600])

In [ ]:
# 4 — Phase-1 models -> phase1_registry.json (same as Phase 2)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach phase1-models)"); continue
    shutil.copy(near(ck[0], "model.py"), f"p1_{name}.py")
    cfgj = json.load(open(near(ck[0], "run_config.json")))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

In [ ]:
# 5 — experiment config helper + runners
TODO = "TODO"
R = "/kaggle/working/p6runs"
def make_cfg(run_name, model, train, data=None, gen=None, kd=None, module="sum_model", dump_teacher=None):
    d = dict(tokenizer_dir="tokenizer", train=P6["train"], test=P6["test"], cache_dir="sum_cache",
             val_cases=TODO,
             val_seed=0,
             max_src_tokens=TODO,
             src_select=TODO,
             max_tgt_tokens=TODO,
             sep_text=TODO,
             start_token="</s>", end_token="</s>")
    d.update(data or {})
    g = dict(max_new_tokens=TODO, min_new_tokens=0, temperature=0.0, top_k=0, top_p=1.0, rep_penalty=1.0,
             no_repeat_ngram=0, batch_size=8, final_val_cases=100, bertscore=False)
    g.update(gen or {})


    cfg = dict(run_name=run_name, out_dir=f"{R}/{run_name}", model_module=module, data=d, model=model, train=train,
               gen=g, kd=kd, dump_teacher_summaries=dump_teacher)
    todo = [k for k, v in {**d, **train, **g, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_on_gpu(cp, gpu):
    lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    subprocess.Popen([sys.executable, "sum_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    print("started", cp, "on GPU", gpu)

def tail_log(name, n=10):
    print("\n".join(open(f"{R}/{name}/train.log").read().splitlines()[-n:]))

def teacher_from(run, new, teacher_cases):
    c = json.load(open(f"{R}/{run}/config.json"))
    p = make_cfg(new, model=c["model"], train=dict(c["train"], epochs=0), data=dict(c["data"], test=None),
                 gen=dict(c["gen"], final_val_cases=0), module=c["model_module"], dump_teacher=teacher_cases)
    cfg = json.load(open(p)); cfg["load_trained"] = f"{R}/{run}/{run}_trained_params.pt"
    json.dump(cfg, open(p, "w"), indent=1); return p

def _cfg_for(job):
    if job.get("teacher_of"):
        return teacher_from(job["teacher_of"], job["name"], job["teacher_cases"])
    kd = None
    if job.get("kd"):
        k = dict(job["kd"]); t = k.pop("teacher"); k.pop("teacher_cases", None)
        kd = dict(teacher_summaries=f"{R}/{t}_teacher/{t}_teacher_teacher_summaries.json", **k)
    return make_cfg(job["name"], job["model"], job["train"], job.get("data"), job.get("gen"), kd=kd,
                    module=job.get("module", "sum_model"))

def run_queues(queues, poll=20, deadline_h=11.25, reserve_min=60):
    import signal
    start = globals().get("NB_START", time.time())
    left_min = lambda: deadline_h * 60 - (time.time() - start) / 60
    qs = {}
    for g, jobs in queues.items():
        lst = []
        for j in jobs:
            if j.get("kd"):
                t = j["kd"]["teacher"]
                lst.append(dict(name=f"{t}_teacher", teacher_of=t, teacher_cases=j["kd"]["teacher_cases"]))
            lst.append(j)
        qs[int(g)] = lst
    cur, t_start = {}, time.time()
    while any(qs.values()) or cur:
        for g in list(qs):
            if g not in cur and qs[g]:
                j = qs[g].pop(0)
                res = j.get("reserve_min", reserve_min)
                if left_min() - res < 10:
                    print(f"SKIPPED {j['name']} on GPU {g}: only {left_min():.0f} min left before the {deadline_h} h guard", flush=True)
                    continue
                try:
                    cp = _cfg_for(j)
                except Exception as e:
                    print(f"SKIPPED {j['name']} on GPU {g}: {type(e).__name__}: {e}", flush=True); continue
                c = json.load(open(cp)); own = float(c["train"].get("max_minutes", 1e9)); cap = left_min() - res
                if cap < own:
                    c["train"]["max_minutes"] = cap; json.dump(c, open(cp, "w"), indent=1)
                    print(f"  {j['name']}: max_minutes {own:.0f} -> {cap:.0f} (12 h guard)", flush=True)
                lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
                p = subprocess.Popen([sys.executable, "sum_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env,
                                     start_new_session=True)
                cur[g] = (p, j["name"], lf, time.time())
                print(f"[{(time.time()-t_start)/60:6.1f} min] started {j['name']} on GPU {g}", flush=True)
        time.sleep(poll)
        if left_min() < -0.35 * 60:
            for g, (p, name, lf, t0) in cur.items():
                if p.poll() is None:
                    print(f"STOPPING {name} on GPU {g}: past the {deadline_h} h guard", flush=True)
                    try: os.killpg(p.pid, signal.SIGKILL)
                    except Exception: p.kill()
            for g in qs: qs[g] = []
        for g, (p, name, lf, t0) in list(cur.items()):
            if p.poll() is not None:
                lf.close(); del cur[g]
                tail = open(f"{R}/{name}/train.log").read().splitlines()[-14:]
                print(f"\n==== [{(time.time()-t_start)/60:6.1f} min] {name} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n"
                      + "\n".join(tail) + "\n", flush=True)

def summary(names):
    for n in names:
        z, res = f"{R}/{n}/{n}_submission.zip", f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res)); v = r.get("val", {})
            print(f"OK  {n:24s} val ROUGE-1 {v.get('rouge1', float('nan')):.4f} | ROUGE-L {v.get('rougeL', float('nan')):.4f} | "
                  f"BERTScore {v.get('bertscore_f1', float('nan')):.4f} | words {v.get('words_pred', float('nan')):.0f} (ref {v.get('words_ref', float('nan')):.0f}) | "
                  f"trainable {r['params_trainable']/1e6:.2f}M of {r['params_total']/1e6:.1f}M | {r['train_minutes']:.1f} min -> {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")

In [ ]:
# 6 — smoke test: dummy model; uncached + cached decoding, ROUGE selection, teacher summaries -> KD student, zip
SM_T = dict(epochs=2, epoch_cases=64, lr=1e-3, batch_size=8, max_minutes=2, val_gen_cases=8)
SM_D = dict(val_cases=40, max_src_tokens=256, src_select="head_tail", max_tgt_tokens=64, sep_text="\n\nSummary:\n")
SM_G = dict(max_new_tokens=24, min_new_tokens=4, final_val_cases=8, batch_size=8, no_repeat_ngram=3, rep_penalty=1.2)
a = make_cfg("smoke_plain", dict(vocab_size=V, cached=False), SM_T, SM_D, SM_G, module="sum_dummy")
b = make_cfg("smoke_cached", dict(vocab_size=V, cached=True), dict(SM_T, select_metric="val_rougeL"),
             dict(SM_D, src_select="last"), dict(SM_G, temperature=0.8, top_p=0.9), module="sum_dummy")
run_on_gpu(a, 0); run_on_gpu(b, 1)
while not all(os.path.exists(f"{R}/{n}/{n}_result.json") or "Traceback" in open(f"{R}/{n}/train.log").read() for n in ("smoke_plain", "smoke_cached")):
    time.sleep(10)
run_queues({0: [dict(name="smoke_kd", model=dict(vocab_size=V, cached=True), train=SM_T, data=SM_D, gen=SM_G, module="sum_dummy",
                     kd=dict(teacher="smoke_cached", teacher_cases=32, mode="add"))]})
for n in ("smoke_plain", "smoke_cached", "smoke_kd"): tail_log(n, 4)
summary(["smoke_plain", "smoke_cached", "smoke_kd"])
assert all(os.path.exists(f"{R}/{n}/{n}_submission.zip") for n in ["smoke_plain", "smoke_cached", "smoke_kd"]), "SMOKE TEST FAILED"
import zipfile; zf = zipfile.ZipFile(f"{R}/smoke_kd/smoke_kd_submission.zip")
print("zip contains:", zf.namelist(), "| first entry:", json.loads(zf.read("prediction.json"))[0])
print("SMOKE TEST PASSED (pipeline only: the dummy model's summaries mean nothing)")

In [ ]:
%%writefile sum_model.py
"""sum_model.py — Phase-6 model."""
import math
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from phase1_loader import load_phase1

BLOCKS = {"prefixlm": "blocks", "encdec": "encoder", "arch4": "blocks", "biscale": "blocks"}
FINAL_NORM = {"prefixlm": "ln_f", "encdec": "enc_ln", "arch4": "ln_f", "biscale": "norm"}
DEFAULT_TARGETS = {
    "prefixlm": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "encdec": ["attn.qkv", "attn.proj", "cross.q", "cross.kv", "cross.proj", "mlp.fc1", "mlp.fc2"],
    "arch4": ["msr.proj", "msr.out", "mlp.fc1", "mlp.fc2"],
    "biscale": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "ffn.gate", "ffn.up", "ffn.down"],
}
ADAPTER_SITES = {
    "prefixlm": ("attn", "mlp"), "encdec": ("attn", "mlp"),
    "arch4": ("msr", "mlp"), "biscale": ("attn", "ffn"),
}


def _first_attr(obj, names):
    for n in names:
        if getattr(obj, n, None) is not None:
            return getattr(obj, n)
    raise AttributeError(f"{type(obj).__name__} has none of {names}")


def _out_splits(backbone_name, path, lin, d_model):
    if path.endswith("attn.qkv"):
        return [lin.out_features // 3] * 3
    if path.endswith("cross.kv"):
        return [lin.out_features // 2] * 2
    if backbone_name == "arch4" and path.endswith("msr.proj"):
        vd = (lin.out_features - 2 * d_model) // 2
        return [d_model, d_model, vd, vd]
    return [lin.out_features]


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, splits, variant):
        super().__init__()
        self.base, self.splits, self.variant = base, list(splits), variant
        self.scale = alpha / r
        self.A = nn.ParameterList()
        self.B = nn.ParameterList()
        for d in self.splits:
            if variant == "plus":
                A = torch.empty(r, d)
                nn.init.kaiming_uniform_(A, a=math.sqrt(5))
                B = torch.zeros(d, r)
            elif variant == "pi":
                A = torch.zeros(r, d)
                A[0, 0] = 1.0
                B = torch.randn(d, r) / math.sqrt(r)
                B[:, 0] = 1.0
            else:
                raise ValueError(f"lorma_variant must be 'plus' or 'pi', got {variant!r}")
            self.A.append(nn.Parameter(A))
            self.B.append(nn.Parameter(B))

    def forward(self, x):
        y = F.linear(x, self.base.weight)
        outs = []
        for i, part in enumerate(y.split(self.splits, dim=-1)):
            A, B = self.A[i], self.B[i]
            if self.variant == "plus":
                outs.append(part + F.linear(F.linear(part, A.to(part.dtype)), B.to(part.dtype)) * self.scale)
            else:
                M = B @ A
                d = M.shape[0]
                idx = torch.arange(d, device=M.device)
                R = torch.gather(M, 1, (idx[None, :] - idx[:, None]) % d)
                outs.append(F.linear(part, R.to(part.dtype)))
        y = torch.cat(outs, dim=-1) if len(outs) > 1 else outs[0]
        if self.base.bias is not None:
            y = y + self.base.bias.to(y.dtype)
        return y


class Adapter(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.adapter = Adapter(d, m)

    def _adapt(self, out):
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)

    def forward(self, *args, **kwargs):
        return self._adapt(self.inner(*args, **kwargs))

    def step(self, *args, **kwargs):
        return self._adapt(self.inner.step(*args, **kwargs))

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["inner"], name)


class AttnContext:
    def __init__(self):
        self.key_mask = None
        self.capture = False


class KVStore:
    def __init__(self, n_layers, max_len):
        self.k, self.v = [None] * n_layers, [None] * n_layers
        self.max_len, self.pos = max_len, 0


class PrefixLMAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, mask, kv=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        if o.rope:
            q, k = self.rope_fn(q, cos.to(q.dtype), sin.to(q.dtype)), self.rope_fn(k, cos.to(q.dtype), sin.to(q.dtype))
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        if kv is not None:
            store, li = kv
            if store.k[li] is None:
                shape = (B, o.h, store.max_len, o.hd)
                store.k[li] = torch.empty(shape, device=k.device, dtype=k.dtype)
                store.v[li] = torch.empty(shape, device=v.device, dtype=v.dtype)
            store.k[li][:, :, start:start + T] = k
            store.v[li][:, :, start:start + T] = v
            k, v = store.k[li][:, :, :start + T], store.v[li][:, :, :start + T]
        if self.pk is not None and not self.ctx.capture:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            if mask is not None:
                mask = torch.cat([mask.new_ones(mask.shape[0], 1, T, n), mask.expand(-1, -1, T, -1)], dim=-1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p_drop if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class EncoderAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, causal=False, cache=None, li=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope_fn(q, cos, sin), self.rope_fn(k, cos, sin)
        km = self.ctx.key_mask
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        elif self.pk is not None:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            if km is not None:
                km = torch.cat([km.new_ones(B, n), km], dim=1)
        mask = None if km is None else km[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class SumModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.name = cfg.get("phase1", "prefixlm")
        if self.name not in BLOCKS:
            raise ValueError(f"phase1 must be one of {list(BLOCKS)}, got {self.name!r}")
        self.method = cfg.get("method", "full")
        backbone, self.info = load_phase1(self.name)
        self.backbone = backbone
        self.p1 = sys.modules[type(backbone).__module__]
        self.ctx = AttnContext()
        self.use_decoder = self.name == "encdec"
        self.grad_ckpt = bool(cfg.get("grad_ckpt", False))

        self.emb_name = next(n for n in ("tok_emb", "embed_tokens", "embed", "wte")
                             if isinstance(getattr(backbone, n, None), nn.Embedding))
        self.d = self.emb.embedding_dim
        self.vocab = self.emb.num_embeddings
        self.head_name = next((n for n, mod in backbone.named_children()
                               if isinstance(mod, nn.Linear) and mod.out_features == self.vocab), None)
        self._make_student()
        if cfg.get("backbone_dropout") is not None:
            for mod in backbone.modules():
                if isinstance(mod, nn.Dropout):
                    mod.p = float(cfg["backbone_dropout"])
        if self.name == "encdec":
            self._wrap_encoder_attention()
            self.backbone.grad_ckpt = self.grad_ckpt
        if self.name == "prefixlm":
            self._wrap_prefixlm_attention()

        self.n_prompt = 0
        self.prompt = None
        self._apply_method()

    @property
    def emb(self):
        return getattr(self.backbone, self.emb_name)

    def lm_head(self, h):
        if self.head_name is not None:
            return getattr(self.backbone, self.head_name)(h)
        return F.linear(h, self.emb.weight)

    def _blocks(self):
        return getattr(self.backbone, BLOCKS[self.name])

    def _stacks(self):
        s = [self._blocks()]
        if self.use_decoder:
            s.append(self.backbone.decoder)
        return s

    def _make_student(self):
        k = self.cfg.get("keep_layers")
        kd = self.cfg.get("keep_dec_layers")
        mode = self.cfg.get("keep_mode", "first")

        def pick(n, k):
            if mode == "spread":
                return sorted({round(i * (n - 1) / max(1, k - 1)) for i in range(k)})
            return list(range(k))

        if k:
            blocks = self._blocks()
            idx = pick(len(blocks), int(k))
            setattr(self.backbone, BLOCKS[self.name], nn.ModuleList([blocks[i] for i in idx]))
            self.kept_layers = idx
        if kd and self.name == "encdec":
            dec = self.backbone.decoder
            idx = pick(len(dec), int(kd))
            self.backbone.decoder = nn.ModuleList([dec[i] for i in idx])
            self.kept_dec_layers = idx

    def _wrap_encoder_attention(self):
        for blk in self.backbone.encoder:
            blk.attn = EncoderAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _wrap_prefixlm_attention(self):
        for blk in self.backbone.blocks:
            blk.attn = PrefixLMAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _apply_method(self):
        cfg, m = self.cfg, self.method
        bb = self.backbone
        if m == "full":
            for p in bb.parameters():
                p.requires_grad = True
            return
        for p in bb.parameters():
            p.requires_grad = False
        if m == "partial":
            n_top = int(cfg.get("unfreeze_layers", 4))
            for stack in self._stacks():
                for blk in list(stack)[-n_top:]:
                    for p in blk.parameters():
                        p.requires_grad = True
            for p in getattr(bb, FINAL_NORM[self.name]).parameters():
                p.requires_grad = True
            if self.use_decoder:
                for p in bb.dec_ln.parameters():
                    p.requires_grad = True
            if cfg.get("unfreeze_embeddings", False):
                self.emb.weight.requires_grad = True
        elif m in ("lora", "lorma"):
            self._wrap_linears(m)
        elif m == "adapter":
            self._add_adapters()
        elif m == "prompt":
            self._add_prompt()
        elif m == "prefix":
            self._add_prefix()
        else:
            raise ValueError(f"method must be lora | lorma | adapter | prompt | prefix (or full | partial), got {m!r}")
        if cfg.get("train_norms", False):
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, nn.GroupNorm)) or type(mod).__name__.lower().endswith("rmsnorm"):
                    for p in mod.parameters(recurse=False):
                        p.requires_grad = True

    def _wrap_linears(self, kind):
        cfg = self.cfg
        targets = cfg.get("targets") or DEFAULT_TARGETS[self.name]
        r = int(cfg.get("rank", 8))
        alpha = float(cfg.get("alpha", 2 * r))
        stack_names = [BLOCKS[self.name]] + (["decoder"] if self.use_decoder else [])
        replaced = []
        for sname in stack_names:
            for bi, blk in enumerate(getattr(self.backbone, sname)):
                for path, mod in list(blk.named_modules()):
                    clean = path.replace("orig.", "").replace("inner.", "")
                    if not isinstance(mod, nn.Linear) or not any(clean == t or clean.endswith("." + t)
                                                                 for t in targets):
                        continue
                    parent_path, _, leaf = path.rpartition(".")
                    parent = blk.get_submodule(parent_path) if parent_path else blk
                    if kind == "lora":
                        new = LoRALinear(mod, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                    else:
                        splits = _out_splits(self.name, clean, mod, self.d) if cfg.get("lorma_split", True) \
                            else [mod.out_features]
                        new = LoRMALinear(mod, r, alpha, splits, cfg.get("lorma_variant", "plus"))
                    setattr(parent, leaf, new)
                    replaced.append(f"{sname}.{bi}.{path}")
        if not replaced:
            raise ValueError(f"no linear matched targets {targets}")
        self.adapted = replaced

    def _add_adapters(self):
        m = int(self.cfg.get("adapter_size", 64))
        style = self.cfg.get("adapter_style", "houlsby")
        attn_site, mlp_site = ADAPTER_SITES[self.name]
        sites = [attn_site, mlp_site] if style == "houlsby" else [mlp_site]
        for stack in self._stacks():
            for blk in stack:
                for s in sites:
                    setattr(blk, s, WithAdapter(getattr(blk, s), self.d, m))

    def _add_prompt(self):
        n = int(self.cfg.get("n_prompt", 20))
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.vocab, (n,), generator=g)
        self.prompt = nn.Parameter(self.emb.weight.detach()[ids].clone().float())
        self.n_prompt = n

    def _add_prefix(self):
        if self.name not in ("prefixlm", "encdec"):
            raise ValueError("prefix tuning is implemented for the attention models prefixlm and encdec only")
        n = int(self.cfg.get("n_prefix", 16))
        for blk in self._blocks():
            blk.attn.add_prefix(n)
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.vocab, (1, n), generator=g)
        dev = self.emb.weight.device
        self.ctx.capture = True
        was = self.training
        self.eval()
        with torch.no_grad():
            x = self._embed(ids.to(dev))
            if self.name == "encdec":
                self._encode(x)
            else:
                self._states_prefixlm(x, n)
        self.ctx.capture = False
        self.train(was)

    def backbone_max_len(self):
        bb = self.backbone
        for n in ("max_seq_len", "max_len"):
            if isinstance(getattr(bb, n, None), int):
                return min(getattr(bb, n), 1024)
        c = getattr(bb, "c", None) or getattr(bb, "cfg", None) or {}
        if isinstance(c, dict) and c.get("max_seq_len"):
            return int(c["max_seq_len"])
        return 1024

    def _run(self, blk, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(blk, *args, use_reentrant=False)
        return blk(*args)

    def _embed(self, ids):
        x = self.emb(ids)
        if self.prompt is not None:
            x = torch.cat([self.prompt.to(x.dtype)[None].expand(ids.shape[0], -1, -1), x], dim=1)
        return x

    def _check_len(self, n, what):
        lim = self.backbone_max_len()
        if n > lim:
            raise ValueError(f"{what} = {n} tokens exceeds the Phase-1 context {lim}")

    def _states_prefixlm(self, x, prefix_len):
        bb = self.backbone
        T = x.shape[1]
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T, device=x.device))
        x = bb.drop(x)
        cos = bb.rope_cos[:T] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T] if bb.rope_sin is not None else None
        plen = torch.full((x.shape[0],), prefix_len, dtype=torch.long, device=x.device)
        mask = self.p1.prefix_lm_mask(T, plen, x.device)
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _states_arch4(self, x):
        bb = self.backbone
        T = x.shape[1]
        t = bb.tables(x.device)
        x = bb.drop(x)
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _states_biscale(self, x):
        bb = self.backbone
        drop = getattr(bb, "drop", None)
        if drop is not None:
            x = drop(x)
        for blk in bb.blocks:
            x = self._run(blk, x)
        return _first_attr(bb, ["norm", "ln_f", "final_norm"])(x)

    def _encode(self, x):
        bb = self.backbone
        T = x.shape[1]
        x = bb.drop(x)
        cos, sin = bb.rope_cos[:T], bb.rope_sin[:T]
        for blk in bb.encoder:
            x = self._run(blk, x, cos, sin)
        return bb.enc_ln(x)

    def logits(self, src, tgt_in):
        B, S = src.shape
        T = tgt_in.shape[1]
        if self.name == "encdec":
            self._check_len(self.n_prompt + S, "prompt + source")
            self._check_len(T, "summary")
            mem = self._encode(self._embed(src))
            return self.backbone.decode(tgt_in, mem)
        P = self.n_prompt
        self._check_len(P + S + T, "prompt + source + summary")
        ids = torch.cat([src, tgt_in], dim=1)
        if self.name == "biscale" and self.prompt is None:
            return self.backbone(ids, torch.full((B,), S, dtype=torch.long, device=ids.device))[:, S:S + T]
        x = self._embed(ids)
        if self.name == "prefixlm":
            h = self._states_prefixlm(x, P + S)
        elif self.name == "arch4":
            h = self._states_arch4(x)
        else:
            h = self._states_biscale(x)
        return self.lm_head(h[:, P + S:P + S + T])

    def forward(self, src, tgt_in):
        return self.logits(src, tgt_in)

    @torch.no_grad()
    def prefill(self, src, tgt_in):
        if self.name == "biscale":
            raise NotImplementedError("Bi-Scale has no cached decoding; the harness recomputes logits() each step")
        bb = self.backbone
        if self.name == "encdec":
            self._check_len(self.n_prompt + src.shape[1], "prompt + source")
            mem = self._encode(self._embed(src))
            cache = self.p1.EncDecCache(len(bb.decoder), bb.max_len)
            for li, blk in enumerate(bb.decoder):
                cache.ck[li], cache.cv[li] = blk.cross.memory_kv(mem)
            return bb._decode_cached(tgt_in, cache), cache
        P, S = self.n_prompt, src.shape[1]
        ids = torch.cat([src, tgt_in], dim=1)
        x = self._embed(ids)
        T0 = x.shape[1]
        self._check_len(T0, "prompt + source + start token")
        if self.name == "arch4":
            t = bb.tables(x.device)
            cos, sin, mask = t["cos"][:T0], t["sin"][:T0], t["mask"][:, :T0, :T0]
            w = t["powers"][:, :T0].flip(-1)
            states = []
            for blk in bb.blocks:
                x, st = blk.forward_state(x, cos, sin, mask, w)
                states.append(st)
            return self.lm_head(bb.ln_f(x[:, -1])), {"states": states, "pos": T0}
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T0, device=x.device))
        cos = bb.rope_cos[:T0] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T0] if bb.rope_sin is not None else None
        plen = torch.full((x.shape[0],), P + S, dtype=torch.long, device=x.device)
        mask = self.p1.prefix_lm_mask(T0, plen, x.device)
        store = KVStore(len(bb.blocks), self.backbone_max_len())
        for li, blk in enumerate(bb.blocks):
            x = blk(x, cos, sin, mask, (store, li), 0)
        store.pos = T0
        return self.lm_head(bb.ln_f(x[:, -1])), store

    @torch.no_grad()
    def decode_step(self, next_ids, cache):
        bb = self.backbone
        if self.name == "encdec":
            return bb._decode_cached(next_ids, cache), cache
        if self.name == "arch4":
            return bb.decode_step(next_ids, cache)
        n = cache.pos
        self._check_len(n + 1, "generated sequence")
        x = self.emb(next_ids)
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(n, n + 1, device=x.device))
        cos = bb.rope_cos[n:n + 1] if bb.rope_cos is not None else None
        sin = bb.rope_sin[n:n + 1] if bb.rope_sin is not None else None
        for li, blk in enumerate(bb.blocks):
            x = blk(x, cos, sin, None, (cache, li), n)
        cache.pos = n + 1
        return self.lm_head(bb.ln_f(x[:, -1])), cache

    def param_groups(self, base_lr):
        new_mult = float(self.cfg.get("peft_lr_mult", 1.0))
        bb_mult = float(self.cfg.get("backbone_lr_mult", 1.0))
        wd = self.cfg.get("weight_decay")
        groups = {"new": [], "new_nd": [], "bb": [], "bb_nd": []}
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            nd = p.dim() < 2 or n == "prompt" or n.endswith((".pk", ".pv"))
            new = (n == "prompt" or n.endswith((".pk", ".pv", ".A", ".B")) or ".A." in n or ".B." in n
                   or ".adapter." in n)
            groups[("new" if new else "bb") + ("_nd" if nd else "")].append(p)
        mult = {"new": new_mult, "bb": bb_mult}
        out = []
        for k, ps in groups.items():
            if not ps:
                continue
            g = {"params": ps, "lr": base_lr * mult[k.split("_")[0]]}
            if k.endswith("_nd"):
                g["weight_decay"] = 0.0
            elif wd is not None:
                g["weight_decay"] = float(wd)
            out.append(g)
        return out


def build_sum_model(model_cfg: dict) -> nn.Module:
    return SumModel(dict(model_cfg))


@torch.no_grad()
def check_init_equivalence(model_cfg: dict, src_len: int = 256, tgt_len: int = 32, batch: int = 2, seed: int = 0):
    adapted = SumModel(dict(model_cfg)).eval()
    plain = SumModel(dict(model_cfg, method="full")).eval()
    dev = adapted.emb.weight.device
    plain.to(dev)
    g = torch.Generator().manual_seed(seed)
    src = torch.randint(1000, adapted.vocab, (batch, src_len), generator=g).to(dev)
    tgt = torch.randint(1000, adapted.vocab, (batch, tgt_len), generator=g).to(dev)
    a, p = adapted.logits(src, tgt).float(), plain.logits(src, tgt).float()
    diff = (a - p).abs().max().item()
    n_all = sum(q.numel() for q in adapted.parameters())
    n_tr = sum(q.numel() for q in adapted.parameters() if q.requires_grad)
    print(f"{model_cfg.get('phase1')}/{model_cfg.get('method')}: max |logits(adapted) - logits(pretrained)| = "
          f"{diff:.2e} (expect ~0 except prompt/prefix) | trainable {n_tr/1e6:.3f}M of {n_all/1e6:.2f}M "
          f"({100*n_tr/n_all:.2f}%)")
    if plain.name in ("prefixlm", "arch4", "biscale") and not model_cfg.get("keep_layers"):
        ids = torch.cat([src, tgt], 1)
        ref = plain.backbone(ids, torch.full((batch,), src_len, dtype=torch.long, device=dev))[:, src_len:].float()
        print(f"  plain logits() vs Phase-1 forward: max diff {(p - ref).abs().max().item():.2e} (expect ~0)")
    if hasattr(adapted, "prefill") and adapted.name != "biscale":
        last, cache = adapted.prefill(src, tgt[:, :1])
        errs = [(last.float() - a[:, 0]).abs().max().item()]
        for t in range(1, tgt_len):
            last, cache = adapted.decode_step(tgt[:, t:t + 1], cache)
            errs.append((last.float() - a[:, t]).abs().max().item())
        print(f"  cache vs logits max diff {max(errs):.2e} over {tgt_len} steps (should be < 1e-2)")
    return diff

In [ ]:
# 8 — PLAN
LAYOUT = "ONE"

T = dict(epochs=10, epoch_cases=3000, lr=5e-4, batch_size=4, grad_accum=4, warmup_frac=0.05, weight_decay=0.01,
         schedule="linear", label_smoothing=0.1, select_metric="val_loss", val_gen_cases=8, max_minutes=100)
D = dict(val_cases=200, max_src_tokens=1019, src_select="head_tail", head_frac=0.6, max_tgt_tokens=896,
         sep_text="\nSummary:\n")
G = dict(max_new_tokens=800, min_new_tokens=300, temperature=0, rep_penalty=1.2, no_repeat_ngram=4,
         batch_size=16, final_val_cases=30, bertscore=False)


D_SHARED = dict(D, max_src_tokens=520, max_tgt_tokens=480)
G_SHARED = dict(G, max_new_tokens=480, min_new_tokens=250)
D_PROMPT = dict(D, max_src_tokens=488, max_tgt_tokens=480)

M_LORMA   = dict(phase1="encdec",   method="lorma", lorma_variant="plus", rank=16, alpha=32, peft_lr_mult=1)
M_KD      = dict(phase1="encdec",   method="full", keep_layers=4, keep_dec_layers=3, keep_mode="spread", backbone_lr_mult=0.1)
M_LORA    = dict(phase1="prefixlm", method="lora", rank=16, alpha=32)
M_PREFIX  = dict(phase1="prefixlm", method="prefix", n_prefix=16, peft_lr_mult=10)
M_HOULSBY = dict(phase1="arch4",    method="adapter", adapter_style="houlsby", adapter_size=64, peft_lr_mult=2)
M_PFEIFF  = dict(phase1="encdec",   method="adapter", adapter_style="pfeiffer", adapter_size=128, peft_lr_mult=2)
M_PROMPT  = dict(phase1="biscale",  method="prompt", n_prompt=32, peft_lr_mult=10, backbone_dropout=0.0)
KD = dict(teacher="r1_lorma_encdec", teacher_cases=800, mode="add")

job = lambda name, m, **kw: dict(name=name, model=m, train=kw.pop("train", T), data=kw.pop("data", D), gen=kw.pop("gen", G), **kw)
QUEUES = {
    "ONE": {0: [job("r1_lorma_encdec", M_LORMA), job("r6_kd_encdec_student", M_KD, kd=KD), job("r4_pfeiffer_encdec", M_PFEIFF)],
            1: [job("r2_lora_prefixlm", M_LORA, data=D_SHARED, gen=G_SHARED),
                job("r5_prefix_prefixlm", M_PREFIX, data=D_SHARED, gen=G_SHARED),
                job("r3_houlsby_arch4", M_HOULSBY, data=D_SHARED, gen=G_SHARED),
                job("r7_prompt_biscale", M_PROMPT, data=D_PROMPT, gen=G_SHARED)]},
    "A":   {0: [job("r1_lorma_encdec", M_LORMA), job("r6_kd_encdec_student", M_KD, kd=KD)],
            1: [job("r2_lora_prefixlm", M_LORA, data=D_SHARED, gen=G_SHARED),
                job("r5_prefix_prefixlm", M_PREFIX, data=D_SHARED, gen=G_SHARED)]},
    "B":   {0: [job("r3_houlsby_arch4", M_HOULSBY, data=D_SHARED, gen=G_SHARED), job("r4_pfeiffer_encdec", M_PFEIFF)],
            1: [job("r7_prompt_biscale", M_PROMPT, data=D_PROMPT, gen=G_SHARED)]},
}
PLAN = QUEUES[LAYOUT]
todo = [(j["name"], k) for js in PLAN.values() for j in js for part in (j["model"], j["train"], j.get("data") or {}, j.get("gen") or {}, j.get("kd") or {})
        for k, v in part.items() if v == TODO or k == "TODO"]
assert not todo, f"fill these first: {todo}"
print("plan:", {g: [j["name"] for j in js] for g, js in PLAN.items()})

In [ ]:
# 9 — model check for every planned model at its REAL sizes (train batch, source, summary): logits shape,

import importlib, sum_model; importlib.reload(sum_model)
for js in PLAN.values():
    for j in js:
        try:
            d, g, tr = j["data"], j["gen"], j["train"]
            sep = len(_tok.encode(d.get("sep_text") or "").ids)
            B_, S_, T_ = int(tr["batch_size"]), int(d["max_src_tokens"]) + sep, int(d["max_tgt_tokens"]) + 1
            m = sum_model.build_sum_model(dict(j["model"])).cuda().train()
            P_ = getattr(m, "n_prompt", 0)
            gen_len = (P_ + S_ + 1 + int(g["max_new_tokens"])) if j["model"]["phase1"] != "encdec" else max(P_ + S_, 1 + int(g["max_new_tokens"]))
            src = torch.randint(5, V, (B_, S_), device="cuda"); tin = torch.randint(5, V, (B_, T_), device="cuda")
            torch.cuda.reset_peak_memory_stats()
            with torch.autocast("cuda", dtype=torch.float16):
                lg = m.logits(src, tin)
            assert tuple(lg.shape) == (B_, T_, V), f"logits must be [B, T, {V}], got {tuple(lg.shape)}"
            lg.float().mean().backward()
            nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            tot, trn = sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)
            msg = (f"OK   {j['name']:22s} trainable {trn/1e6:.3f}M of {tot/1e6:.2f}M ({100*trn/tot:.2f}%) | "
                   f"train step B{B_} src {S_} (sep {sep}) tgt {T_}: peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB | "
                   f"generation length {gen_len}" + ("  <-- OVER 1024" if gen_len > 1024 else "") + f" | no-grad {nog[:3]}")
            del lg; m.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            has_cache = hasattr(m, "prefill") and hasattr(m, "decode_step")
            if has_cache:
                m.eval()
                try:
                    with torch.no_grad():
                        s2, t2 = src[:2], tin[:2, :4]
                        full = m.logits(s2, t2).float()
                        a, cache_ = m.prefill(s2, t2[:, :1]); dmax = (a.float() - full[:, 0]).abs().max().item()
                        for t in range(1, 4):
                            a, cache_ = m.decode_step(t2[:, t:t+1], cache_); dmax = max(dmax, (a.float() - full[:, t]).abs().max().item())
                    msg += f" | cache vs logits {dmax:.2e}" + ("" if dmax < 1e-2 else "  <-- MISMATCH")
                except NotImplementedError:
                    has_cache = False
            if not has_cache:
                msg += " | no cache (generation recomputes; slower)"
            print(msg, flush=True)
            del m
        except Exception as e:
            print(f"FAIL {j['name']:22s} {type(e).__name__}: {str(e)[:300]}", flush=True)
        torch.cuda.empty_cache()

In [ ]:
# 10 — RUN THE PLAN (blocks until every queue on this account is done; do NOT stop this cell while jobs are queued)
run_queues(PLAN)
summary([j["name"] for js in PLAN.values() for j in js])

In [ ]:
# 11 — results table (report) + download links + small backup zip (no weights)
import pandas as pd
from IPython.display import FileLink, display
rows = []
for rp in sorted(glob.glob(f"{R}/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke") or r["run"].endswith("_teacher"): continue
    v = r.get("val", {})
    rows.append(dict(run=r["run"], val_rouge1=v.get("rouge1"), val_rouge2=v.get("rouge2"), val_rougeL=v.get("rougeL"),
                     val_bertscore_f1=v.get("bertscore_f1"), words_pred=v.get("words_pred"), words_ref=v.get("words_ref"),
                     trainable_M=r["params_trainable"]/1e6, total_M=r["params_total"]/1e6, train_min=r["train_minutes"],
                     peak_GB=r["peak_mem_gb"], epochs_done=len(r.get("history", [])), max_src=r["max_src_tokens"],
                     src_select=r["src_select"], max_tgt=r["max_tgt_tokens"], test_gen_min=(r.get("test_gen_seconds") or 0)/60))
df = pd.DataFrame(rows).sort_values("run"); display(df)
df.to_csv(f"{R}/results.csv", index=False)
os.chdir("/kaggle/working")
for n in df.run: display(FileLink(f"p6runs/{n}/{n}_submission.zip"))
display(FileLink("p6runs/results.csv"))
!zip -qr p6_backup.zip sum_model.py p6runs -x "*.pt" "*smoke*"
display(FileLink("p6_backup.zip"))